# Single Item Solver Proof

This notebook proves the first layer of the production `solve_order()` path using the original v1 dataset. It deliberately focuses on **one physical item** before we move to multi-item placement.

The proof checks, in order:

1. physical item count and total item volume;
2. candidate cartons sorted by external volume, smallest first;
3. weight and usable-volume eligibility;
4. every allowed orientation under `VerticalRotation`;
5. the first geometrically feasible carton;
6. the independently validated solver result.

The sample is reproducible: seed `26` randomly selects one record from the v1 records containing exactly one physical item.

In [ ]:
from pathlib import Path
import json
import random
import sys

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))

from bin_packing_3d import solve_order


In [ ]:
with open(ROOT / 'data' / 'raw' / 'data_sample_v1.json', encoding='utf-8') as f:
    records = json.load(f)

single_item_records = [
    record for record in records
    if sum(item['Quantity'] for item in record['input']['Items']['ItemsList']) == 1
]

sample = random.Random(26).choice(single_item_records)
request = sample['input']
item = request['Items']['ItemsList'][0]
boxes = request['Bins']['BinsList']

print('Randomly selected OrderId:', request['OrderId'])
print('Item:', item)
print('Candidate boxes by external volume:')
for box in sorted(boxes, key=lambda b: b['Length'] * b['Width'] * b['Height']):
    volume = box['Length'] * box['Width'] * box['Height']
    print(f"  {box['Code']}: {volume:g} mm^3")


## Run through the production API with tracing

`trace=True` only exposes decisions. It does not change the packing strategy or bypass validation.

In [ ]:
parameters = request['Bins']['Parameters']
config = {
    'strategy': 'first_fit',
    'high_item_count_threshold': parameters['BinMaxFillCheckMinItemQty'],
    'high_item_count_max_fill_pct': parameters['BinMaxFillPct'],
    'bin_buffer': parameters['BinBuffer'],
}

result = solve_order(
    order=request,
    boxes=boxes,
    config=config,
    trace=True,
)


In [ ]:
print('\n=== Returned result ===')
print('Status:', result.status)
print('Selected carton:', result.packed_boxes[0].box_code)
print('Validation passed:', result.validation.valid)
print('Placement:', result.placements[0])


## What this proves

For the seeded v1 sample, Order 797 contains one `330 × 215 × 80 mm` item with `VerticalRotation = 1`.

Box2 is rejected before geometry because the item's `5,676,000 mm³` volume exceeds Box2's usable volume after the configured 6 mm height buffer. Box3 has enough usable volume, but no allowed orientation can fit its dimensions. Box4 is therefore the first feasible carton in volume-sorted order and is selected.

The final `PackingResult` is then independently validated. This is the single-item proof only. Multi-item coexistence, candidate points, overlap and placement scoring should be demonstrated separately rather than hidden inside this first notebook.